In [1]:
import torch
from yolox.exp import get_exp
import os

# --- Configuration ---
# Path to your YOLOX checkpoint file (.pth)
ckpt_path = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"

# Path to your YOLOX experiment file (e.g., yolox_nano_cid.py).
exp_file = "exps/default/yolox_nano_cid.py"

# Input image dimensions (Height, Width) - must match your training input size.
input_height = 320
input_width = 320

# --- Pre-checks ---
if not os.path.exists(ckpt_path):
    print(f"Error: Checkpoint file not found at '{ckpt_path}'")
    print("Please ensure the 'YOLOX_outputs/yolox_nano_cid/' directory exists and 'best_ckpt.pth' is inside it.")
    exit()

if not os.path.exists(exp_file):
    print(f"Error: Experiment file not found at '{exp_file}'")
    print("Please ensure the 'exps/default/' directory exists and your experiment file is inside it.")
    print("You might need to adjust 'exp_file' variable to point to your specific YOLOX experiment configuration.")
    exit()

print(f"Inspecting raw output of YOLOX .pth model...")
print(f"Loading model from checkpoint: {ckpt_path}")
print(f"Using experiment file: {exp_file}")
print(f"Using dummy input size: {input_width}x{input_height}")

try:
    # 1. Load YOLOX model architecture
    exp = get_exp(exp_file, None)
    model = exp.get_model()

    # 2. Load checkpoint weights
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    if "model" in ckpt:
        model.load_state_dict(ckpt["model"])
    else:
        model.load_state_dict(ckpt)

    # 3. Set the model to evaluation mode
    model.eval()

    # 4. Create a dummy input tensor (NCHW format)
    dummy_input = torch.randn(1, 3, input_height, input_width)
    print(f"Created dummy input tensor with shape: {dummy_input.shape}")

    # 5. Get feature maps from the backbone
    with torch.no_grad(): # Disable gradient calculation for inference
        fpn_outs = model.backbone(dummy_input)
    print(f"\nBackbone output (fpn_outs) consists of {len(fpn_outs)} tensors:")
    for i, fpn_out in enumerate(fpn_outs):
        print(f"  FPN Output {i} shape: {fpn_out.shape}")

    # 6. Set decode_in_inference to False on the head
    # This ensures the head returns raw predictions, not decoded boxes with NMS.
    model.head.decode_in_inference = False
    print("\nModel head set to decode_in_inference = False.")

    # 7. Get raw predictions from the head
    with torch.no_grad():
        head_output = model.head(fpn_outs)

    print("\n--- YOLOX Head Raw Output Details ---")
    # YOLOXHead.forward when decode_in_inference=False typically returns a tuple:
    # (raw_prediction_tensors_list, x_shifts, y_shifts, expanded_strides)
    # Or, in some versions, it might directly return a single concatenated tensor.

    if isinstance(head_output, tuple):
        # Assuming it's (raw_prediction_tensors_list, x_shifts, y_shifts, expanded_strides)
        raw_prediction_tensors_list = head_output[0]
        print(f"Head output is a tuple. First element (raw predictions) is a list of {len(raw_prediction_tensors_list)} tensors:")
        for i, tensor in enumerate(raw_prediction_tensors_list):
            print(f"  Raw Prediction Tensor {i} shape: {tensor.shape}")
        
        # Also print shapes of auxiliary outputs if they exist
        if len(head_output) > 1:
            print(f"  x_shifts shape: {head_output[1].shape}")
            print(f"  y_shifts shape: {head_output[2].shape}")
            print(f"  expanded_strides shape: {head_output[3].shape}")

        # Now, let's manually perform the flattening and concatenation as we tried for ONNX
        processed_outputs = []
        for output_tensor in raw_prediction_tensors_list:
            # output_tensor shape: [batch_size, (5 + num_classes), H, W]
            flattened_tensor = output_tensor.flatten(2) # [batch_size, (5 + num_classes), H*W]
            transposed_tensor = flattened_tensor.permute(0, 2, 1) # [batch_size, H*W, (5 + num_classes)]
            processed_outputs.append(transposed_tensor)
        
        final_concatenated_output = torch.cat(processed_outputs, dim=1)
        print(f"\nManually concatenated output shape: {final_concatenated_output.shape}")

    elif isinstance(head_output, torch.Tensor):
        # If it's already a single tensor
        print(f"Head output is a single tensor with shape: {head_output.shape}")
        # Let's try to permute it to the desired format if it's not already
        if head_output.shape[-1] != (5 + exp.num_classes):
            print(f"  Attempting to permute to [batch_size, total_predictions, (5 + num_classes)]...")
            permuted_output = head_output.permute(0, 2, 1)
            print(f"  Permuted output shape: {permuted_output.shape}")
        else:
            print(f"  Output already in [batch_size, total_predictions, (5 + num_classes)] format.")

    else:
        print(f"Unexpected head output type: {type(head_output)}")

    print("\nInspection complete.")

except Exception as e:
    print(f"\nAn error occurred during model inspection: {e}")
    print("Please ensure:")
    print("1. You have the YOLOX repository correctly installed and accessible.")
    print("2. The 'exp_file' variable points to the correct experiment configuration file for your model.")
    print("3. The 'ckpt_path' variable points to your actual checkpoint file.")
    print("4. Your PyTorch and YOLOX installations are compatible.")



Inspecting raw output of YOLOX .pth model...
Loading model from checkpoint: YOLOX_outputs/yolox_nano_cid/best_ckpt.pth
Using experiment file: exps/default/yolox_nano_cid.py
Using dummy input size: 320x320
Created dummy input tensor with shape: torch.Size([1, 3, 320, 320])

Backbone output (fpn_outs) consists of 3 tensors:
  FPN Output 0 shape: torch.Size([1, 64, 40, 40])
  FPN Output 1 shape: torch.Size([1, 128, 20, 20])
  FPN Output 2 shape: torch.Size([1, 256, 10, 10])

Model head set to decode_in_inference = False.

--- YOLOX Head Raw Output Details ---
Head output is a single tensor with shape: torch.Size([1, 2100, 7])
  Output already in [batch_size, total_predictions, (5 + num_classes)] format.

Inspection complete.


In [6]:
import torch
import torch.nn as nn
from yolox.exp import get_exp
import os

# --- Configuration ---
# Path to your YOLOX checkpoint file (.pth)
# Ensure this path is correct.
ckpt_path = "YOLOX_outputs/yolox_nano_cid/best_ckpt.pth"

# Path to your YOLOX experiment file (e.g., yolox_nano_cid.py).
# This file defines the model architecture.
# IMPORTANT: Update this to the correct path of your experiment file.
# Based on your previous output, it seems to be 'exps/default/yolox_nano_cid.py'.
exp_file = "exps/default/yolox_nano_cid.py"

# Path for the output ONNX model.
onnx_path = "models/yolox_nano_cid_raw_output.onnx"

# Input image dimensions (Height, Width) - must match your training input size.
# Based on your previous tensor details, 320x320 is used.
input_height = 320
input_width = 320

# Number of classes your model was trained on
num_classes = 2 # IMPORTANT: Set this to your actual number of classes

# --- Pre-checks ---
if not os.path.exists(ckpt_path):
    print(f"Error: Checkpoint file not found at '{ckpt_path}'")
    print("Please ensure the 'YOLOX_outputs/yolox_nano_cid/' directory exists and 'best_ckpt.pth' is inside it.")
    exit()

if not os.path.exists(exp_file):
    print(f"Error: Experiment file not found at '{exp_file}'")
    print("Please ensure the 'exps/default/' directory exists and your experiment file is inside it.")
    print("You might need to adjust 'exp_file' variable to point to your specific YOLOX experiment configuration.")
    exit()

# Create the 'models' directory if it doesn't exist
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

print(f"Starting YOLOX .pth to ONNX conversion...")
print(f"Loading model from checkpoint: {ckpt_path}")
print(f"Using experiment file: {exp_file}")
print(f"Output ONNX path: {onnx_path}")
print(f"Expected input size: {input_width}x{input_height}")

# --- Updated: Wrapper Module for ONNX Export with Fixed Output Shape ---
class ExportModel(nn.Module):
    def __init__(self, model, num_classes, input_height, input_width):
        super().__init__()
        self.backbone = model.backbone
        self.head = model.head
        self.num_classes = num_classes
        self.input_height = input_height
        self.input_width = input_width
        # Crucial: Ensure the head is configured for raw output during inference
        self.head.decode_in_inference = False

        # Calculate expected static dimensions for total_predictions and attributes_per_prediction
        # These must be fixed values for ONNX export and are calculated once.
        self.total_predictions_fixed = (self.input_height // 8 * self.input_width // 8) + \
                                       (self.input_height // 16 * self.input_width // 16) + \
                                       (self.input_height // 32 * self.input_width // 32)
        self.attributes_per_prediction_fixed = 5 + self.num_classes

    def forward(self, x):
        # Pass input through the backbone to get feature maps
        fpn_outs = self.backbone(x)

        # Pass feature maps to the head to get raw predictions.
        # This should return the single concatenated tensor.
        raw_output_tensor = self.head(fpn_outs)
        
        # Explicitly use .view() to force a static shape for ONNX.
        # The batch size is kept dynamic using x.shape[0].
        # All other dimensions are fixed integers.
        final_output = raw_output_tensor.view(x.shape[0], self.total_predictions_fixed, self.attributes_per_prediction_fixed)
            
        return final_output

try:
    # 1. Load YOLOX model architecture from the experiment file
    exp = get_exp(exp_file, None)
    print(f"Experiment loaded successfully. Number of classes: {exp.num_classes}")

    # Get the model instance based on the experiment configuration
    model = exp.get_model()
    print(f"Model architecture obtained from experiment.")

    # 2. Load checkpoint weights
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    if "model" in ckpt:
        model.load_state_dict(ckpt["model"])
        print("Model weights loaded from checkpoint.")
    else:
        model.load_state_dict(ckpt)
        print("Model weights loaded directly (no 'model' key found in checkpoint).")

    # 3. Set the original model to evaluation mode
    model.eval()
    print("Original model set to evaluation mode.")

    # 4. Instantiate the updated wrapper model
    # Pass input_height and input_width to the ExportModel
    export_model = ExportModel(model, num_classes=exp.num_classes, 
                               input_height=input_height, input_width=input_width)
    # Set the wrapper model to evaluation mode as well
    export_model.eval()
    print("Created ExportModel wrapper for ONNX conversion and set to evaluation mode.")

    # 5. Create a dummy input tensor
    # YOLOX models expect NCHW format (Batch, Channels, Height, Width).
    dummy_input = torch.randn(1, 3, input_height, input_width)
    print(f"Created dummy input tensor with shape: {dummy_input.shape}")

    # 6. Export to ONNX using the wrapper model
    # The ExportModel.forward is now expected to return a single tensor with a fixed shape.
    output_names = ["output"]
    print(f"Exporting with output names: {output_names}")

    torch.onnx.export(
        export_model, # Use the wrapped model for export
        dummy_input,
        onnx_path,
        input_names=["input"],
        output_names=output_names,
        opset_version=11,
        # Define the fixed output dimensions for the ONNX graph.
        # The output shape should now be [batch_size, total_predictions, attributes_per_prediction].
        dynamic_axes={"input": {0: "batch_size"},
                      "output": {0: "batch_size"}}, # Only batch size is dynamic
        verbose=False, # Set to True for more detailed export logs
    )

    print(f"\nSuccessfully exported YOLOX model to ONNX at: {onnx_path}")

except Exception as e:
    print(f"\nAn error occurred during ONNX export: {e}")
    print("Please ensure:")
    print("1. You have the YOLOX repository correctly installed and accessible.")
    print("2. The 'exp_file' variable points to the correct experiment configuration file for your model.")
    print("3. The 'ckpt_path' variable points to your actual checkpoint file.")
    print("4. Your PyTorch and YOLOX installations are compatible.")
    print("5. The input_height and input_width match your model's expected input dimensions.")
    print("6. The 'num_classes' variable in the script matches your model's actual number of classes.")



Starting YOLOX .pth to ONNX conversion...
Loading model from checkpoint: YOLOX_outputs/yolox_nano_cid/best_ckpt.pth
Using experiment file: exps/default/yolox_nano_cid.py
Output ONNX path: models/yolox_nano_cid_raw_output.onnx
Expected input size: 320x320
Experiment loaded successfully. Number of classes: 2
Model architecture obtained from experiment.
Model weights loaded from checkpoint.
Original model set to evaluation mode.
Created ExportModel wrapper for ONNX conversion and set to evaluation mode.
Created dummy input tensor with shape: torch.Size([1, 3, 320, 320])
Exporting with output names: ['output']

Successfully exported YOLOX model to ONNX at: models/yolox_nano_cid_raw_output.onnx


In [4]:
import onnx
import numpy as np
import os
import onnx.helper # Import onnx.helper to use tensor_dtype_to_np_dtype

# --- Configuration ---
# Path to your ONNX model file.
# This should be the output from the .pth to ONNX conversion.
onnx_model_path = "models/best_ckpt.onnx"

# --- Pre-checks ---
if not os.path.exists(onnx_model_path):
    print(f"Error: ONNX model file not found at '{onnx_model_path}'")
    print("Please ensure the ONNX conversion was successful and the file exists.")
    exit()

print(f"Inspecting ONNX model: {onnx_model_path}")

try:
    # 1. Load the ONNX model
    model = onnx.load(onnx_model_path)
    graph = model.graph

    # 2. Get input tensor details
    print("\n--- ONNX Input Tensor Details ---")
    for input_node in graph.input:
        print(f"  Name: {input_node.name}")
        # Get shape from input_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in input_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        # Use onnx.helper.tensor_dtype_to_np_dtype to avoid DeprecationWarning
        print(f"  Dtype: {onnx.helper.tensor_dtype_to_np_dtype(input_node.type.tensor_type.elem_type)}")
        print("-" * 30)

    # 3. Get output tensor details
    print("\n--- ONNX Output Tensor Details ---")
    for output_node in graph.output:
        print(f"  Name: {output_node.name}")
        # Get shape from output_node.type.tensor_type.shape.dim
        shape = [dim.dim_value if dim.dim_value != 0 else -1 for dim in output_node.type.tensor_type.shape.dim]
        print(f"  Shape: {shape}")
        # Use onnx.helper.tensor_dtype_to_np_dtype to avoid DeprecationWarning
        print(f"  Dtype: {onnx.helper.tensor_dtype_to_np_dtype(output_node.type.tensor_type.elem_type)}")
        print("-" * 30)

    print("\nONNX model inspection complete.")

except Exception as e:
    print(f"\nAn error occurred during ONNX model inspection: {e}")
    print("Please ensure:")
    print("1. The ONNX model file at '{onnx_model_path}' is valid.")
    print("2. You have `onnx` installed (`pip install onnx`).")



Inspecting ONNX model: models/best_ckpt.onnx

--- ONNX Input Tensor Details ---
  Name: images
  Shape: [1, 3, 640, 640]
  Dtype: float32
------------------------------

--- ONNX Output Tensor Details ---
  Name: output
  Shape: [1, 8400, 7]
  Dtype: float32
------------------------------

ONNX model inspection complete.


In [31]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf
import os
import shutil # For cleaning up directories
import numpy as np # For creating dummy data

# --- Configuration ---
# Path to your input ONNX model.
onnx_path = "models/yolox_nano_cid_raw_output.onnx"

# Path for the output TFLite model.
tflite_path = "models/yolox_nano_cid.tflite"

# Define the expected input shape for the model (Batch, Channels, Height, Width)
# This should match the dummy input used during ONNX export.
input_shape = [1, 3, 320, 320] # IMPORTANT: Ensure this matches your model's input

# Define the expected output tensor name from the TFLite inspection.
# This is 'Identity'.
output_tensor_name = "Identity"

# --- Pre-checks ---
if not os.path.exists(onnx_path):
    print(f"Error: ONNX file not found at '{onnx_path}'")
    print("Please ensure the ONNX conversion from .pth was successful and the file exists.")
    exit()

# Create the 'models' directory if it doesn't exist
os.makedirs(os.path.dirname(tflite_path), exist_ok=True)

print(f"Starting ONNX to TFLite conversion...")
print(f"Input ONNX path: {onnx_path}")
print(f"Output TFLite path: {tflite_path}")
print(f"Using explicit input shape for TFLite converter: {input_shape}")
print(f"Explicitly setting output tensor name for TFLite converter: '{output_tensor_name}'")


saved_model_dir = "temp_tf_savedmodel"

try:
    # 1. Load the ONNX model
    print("Loading ONNX model...")
    onnx_model = onnx.load(onnx_path)
    print("ONNX model loaded successfully.")

    # 2. Prepare the ONNX model for TensorFlow and export as SavedModel
    print("Converting ONNX model to TensorFlow SavedModel...")
    tf_rep = prepare(onnx_model)
    
    # Clean up previous temp directory if it exists
    if os.path.exists(saved_model_dir):
        shutil.rmtree(saved_model_dir)
    tf_rep.export_graph(saved_model_dir)
    print(f"TensorFlow SavedModel exported to: {saved_model_dir}")

    # --- Reverting to simpler from_saved_model API ---
    print("Converting TensorFlow SavedModel to TFLite using from_saved_model API...")
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)

    # Apply default optimizations
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    # Enable Select TF Ops (still crucial)
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]

    # Explicitly set the output arrays.
    converter.output_arrays = [output_tensor_name]

    # Remove representative_dataset for now, as it's primarily for quantization
    # and doesn't seem to resolve shape issues in this float model.
    # def representative_dataset_gen():
    #     yield [np.zeros(input_shape, dtype=np.float32)]
    # converter.representative_dataset = representative_dataset_gen

    tflite_model = converter.convert()
    print("TFLite model conversion complete.")

    # 4. Save the TFLite model
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    print(f"\nSuccessfully exported TFLite model to: {tflite_path}")

except Exception as e:
    print(f"\nAn error occurred during ONNX to TFLite export: {e}")
    print("Please ensure:")
    print("1. The ONNX model at '{onnx_path}' is valid.")
    print("2. You have `onnx-tf` and `tensorflow` installed (`pip install onnx-tf tensorflow`).")
    print("3. The ONNX model does not contain operations that are fundamentally unsupported by TensorFlow Lite.")
    print("   (Often, complex post-processing like NMS is best handled client-side on mobile.)")
    print("4. The 'input_shape' variable in the script matches your model's expected input dimensions.")
    print("5. The 'output_tensor_name' variable matches the actual output name in the TFLite model (currently 'Identity').")

finally:
    # Clean up the temporary SavedModel directory
    if os.path.exists(saved_model_dir):
        shutil.rmtree(saved_model_dir)
        print(f"Cleaned up temporary SavedModel directory: {saved_model_dir}")



Starting ONNX to TFLite conversion...
Input ONNX path: models/yolox_nano_cid_raw_output.onnx
Output TFLite path: models/yolox_nano_cid.tflite
Using explicit input shape for TFLite converter: [1, 3, 320, 320]
Explicitly setting output tensor name for TFLite converter: 'Identity'
Loading ONNX model...
ONNX model loaded successfully.
Converting ONNX model to TensorFlow SavedModel...


INFO:absl:Function `__call__` contains input name(s) x, y with unsupported characters which will be renamed to transpose_343_x, add_97_y in the SavedModel.
INFO:absl:Found untraced functions such as gen_tensor_dict while saving (showing 1 of 1). These functions will not be directly callable after loading.


INFO:tensorflow:Assets written to: temp_tf_savedmodel\assets


INFO:tensorflow:Assets written to: temp_tf_savedmodel\assets
INFO:absl:Writing fingerprint to temp_tf_savedmodel\fingerprint.pb


TensorFlow SavedModel exported to: temp_tf_savedmodel
Converting TensorFlow SavedModel to TFLite using from_saved_model API...
TFLite model conversion complete.

Successfully exported TFLite model to: models/yolox_nano_cid.tflite
Cleaned up temporary SavedModel directory: temp_tf_savedmodel


In [32]:
import tensorflow as tf
import numpy as np
import os

# --- Configuration ---
# Path to your TFLite model file.
tflite_model_path = "models/yolox_nano_cid.tflite"

# --- Pre-checks ---
if not os.path.exists(tflite_model_path):
    print(f"Error: TFLite model file not found at '{tflite_model_path}'")
    print("Please ensure the TFLite conversion was successful and the file exists.")
    exit()

print(f"Inspecting TFLite model: {tflite_model_path}")

try:
    # 1. Load the TFLite model and allocate tensors.
    interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
    interpreter.allocate_tensors()

    # 2. Get input tensor details.
    input_details = interpreter.get_input_details()
    print("\n--- Input Tensor Details ---")
    for i, detail in enumerate(input_details):
        print(f"Input Tensor {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Index: {detail['index']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Shape Signature: {detail['shape_signature']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")
        print(f"  Quantization Parameters: {detail['quantization_parameters']}")
        print(f"  Sparsity Parameters: {detail['sparsity_parameters']}")
        print("-" * 30)

    # 3. Get output tensor details.
    output_details = interpreter.get_output_details()
    print("\n--- Output Tensor Details ---")
    for i, detail in enumerate(output_details):
        print(f"Output Tensor {i}:")
        print(f"  Name: {detail['name']}")
        print(f"  Index: {detail['index']}")
        print(f"  Shape: {detail['shape']}")
        print(f"  Shape Signature: {detail['shape_signature']}")
        print(f"  Dtype: {detail['dtype']}")
        print(f"  Quantization: {detail['quantization']}")
        print(f"  Quantization Parameters: {detail['quantization_parameters']}")
        print(f"  Sparsity Parameters: {detail['sparsity_parameters']}")
        print("-" * 30)

    print("\nInspection complete.")

except Exception as e:
    print(f"\nAn error occurred during TFLite model inspection: {e}")
    print("Please ensure:")
    print("1. The TFLite model file at '{tflite_model_path}' is valid and not corrupted.")
    print("2. You have TensorFlow installed (`pip install tensorflow`).")



Inspecting TFLite model: models/yolox_nano_cid.tflite

--- Input Tensor Details ---
Input Tensor 0:
  Name: serving_default_input:0
  Index: 0
  Shape: [  1   3 320 320]
  Shape Signature: [ -1   3 320 320]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
  Quantization Parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
  Sparsity Parameters: {}
------------------------------

--- Output Tensor Details ---
Output Tensor 0:
  Name: StatefulPartitionedCall:0
  Index: 992
  Shape: [1 1 1]
  Shape Signature: [-1 -1 -1]
  Dtype: <class 'numpy.float32'>
  Quantization: (0.0, 0)
  Quantization Parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}
  Sparsity Parameters: {}
------------------------------

Inspection complete.
